# Time Series Merging v4 — DuckDB Range Join Pipeline

Builds an aggregated sensor feature table from 7 raw sensor `.txt` files and 7 cleaned Events
parquet files, then merges the result with the 3 SAP base tables (IP1, IP2, IP3).

**v4 adds sub-event sensor feature extraction (Cell 5c/5d):** sensor readings scoped to Child 2
sub-step time windows, not just the full batch process window.

**Key improvement over v1:** replaces the O(n_events × n_sensor_rows) `iterrows()` loop
with a DuckDB range join, reducing runtime from ~5 days to ~15–30 minutes.

Pipeline:
1. Load SAP cleaned files → build `VALID_BATCH_IDS`
2. Clean and save 7 Events `.xlsx` files as parquet
3. Aggregate events duration features per batch per phase (count, total, mean, std, max)
4. Extract Child 1/Child 2 sub-event duration features (Cell 5b)
5. Build child2_xto_map using Phase-level event rows (Cell 5c)
6. Run second DuckDB pass — sub-event sensor features (Cell 5d)
7. Build XTO→Batch time-window map, filtered to valid SAP batches (Cell 6)
8. Process each sensor `.txt` file via DuckDB range join → aggregate (Cell 7)
9. Combine, pivot to wide (PG / DEAE / PREP columns), merge all feature sets, save

## Cell 1 — Imports and Install

In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'duckdb', '--quiet'])

import os
import re
import warnings
import gc
import csv
from pathlib import Path
from functools import reduce

import numpy as np
import pandas as pd
from tqdm import tqdm
import duckdb
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import joblib

warnings.filterwarnings('ignore')
print(f'DuckDB version: {duckdb.__version__}')
print('All imports successful.')

DuckDB version: 1.5.2
All imports successful.


## Cell 2 — Config and Paths

In [ ]:
DEV_MODE = False

CHUNKSIZE   = 10_000  if DEV_MODE else 100_000
SAMPLE_ROWS = 1_000   if DEV_MODE else None

PROJECT_ROOT       = Path(r"C:\Hackathon-GSK\data\raw")
DATA_DIR           = PROJECT_ROOT / 'files'
EVENTS_DIR         = DATA_DIR / 'Events'
SAP_DIR            = DATA_DIR / 'SAP ZQM105'
SENSOR_DIR         = DATA_DIR / 'Sensor'
CLEANED_DIR        = Path(r"C:\Hackathon-GSK\data\processed\cleaned_data")
EVENTS_CLEANED_DIR = CLEANED_DIR / 'Events'
SAP_CLEANED_DIR    = CLEANED_DIR / 'SAP'
OUTPUT_DIR         = Path(r"C:\Hackathon-GSK\Final_Submission\outputs\basetable")

# Maps each events phase key to its process stage name (confirmed by GSK Philippe)
PHASE_TO_STAGE = {
    'pg_up4'     : 'PG',
    'deae_up12'  : 'DEAE',
    'pg_deae_up1': 'PREP',
    'pg_deae_up2': 'PREP',
    'pg_up3'     : 'PREP',
    'pg_deae_up5': 'PREP',
    'pg_up6'     : 'PREP',
}

SENSOR_NUMERIC_COLS = [
    'AI_A1', 'AI_A2', 'AI_A3',
    'Accumulated Volume',
    'FIC_C11',
    'MT_I1_OUT', 'MT_I2_OUT',
    'PIC_A1', 'PIC_I1', 'PIC_I1_OUT', 'PI_A2',
    'TA_A2_ASYM', 'TA_A2_HETP',
    'TA_O1_ASYM', 'TA_O1_HETP',
    'TI_A1', 'TI_A2',
    'UV Light',
    'XV-O1_ZSO',
    'RunCalc_UV_DuringHarvest',
    'RunCalc_UV_ForIntegral_v2',
]

SENSOR_BOOL_COLS = ['UV Elution']

print(f'DEV_MODE:           {DEV_MODE}')
print(f'CHUNKSIZE:          {CHUNKSIZE}')
print(f'Sensor numeric cols:{len(SENSOR_NUMERIC_COLS)}')
print(f'Stages mapped:      {set(PHASE_TO_STAGE.values())}')

DEV_MODE:           False
CHUNKSIZE:          100000
Sensor numeric cols:21
Stages mapped:      {'PREP', 'DEAE', 'PG'}


## Cell 3 — Load SAP Files and Build Valid Batch ID List

Only batches present in at least one of IP1/IP2/IP3 will be processed downstream.
This reduces the XTO map from ~1,734 batches to ~207 valid SAP batches.

In [ ]:
ip1 = pd.read_csv("C:\Hackathon-GSK\Final_Submission\data\cleaned_data\SAP\260109_ZQM105_IP1_P030_PURIF_cleanedApr28.csv")
ip2 = pd.read_csv("C:\Hackathon-GSK\Final_Submission\data\cleaned_data\SAP\260109_ZQM105_IP2_P030_PURIF_cleanedApr28.csv")
ip3 = pd.read_csv("C:\Hackathon-GSK\Final_Submission\data\cleaned_data\SAP\260109_ZQM105_IP3_P030_PURIF_cleanedApr28.csv")

VALID_BATCH_IDS = set(
    pd.concat([ip1['Batch'], ip2['Batch'], ip3['Batch']]).dropna().unique()
)

print(f'IP1 shape: {ip1.shape}')
print(f'IP2 shape: {ip2.shape}')
print(f'IP3 shape: {ip3.shape}')
print(f'Valid SAP batch IDs: {len(VALID_BATCH_IDS)}')

IP1 shape: (90, 147)
IP2 shape: (30, 133)
IP3 shape: (87, 144)
Valid SAP batch IDs: 207


## Cell 4 — Load and Clean Events Files

Applies four cleaning rules to all 7 Events `.xlsx` files (UF excluded):
1. Forward-fill `BatchID` — blank cells inherit the preceding non-blank ID
2. **Normalize `BatchID`** — strip stage suffix to match SAP format
   (`3_hash_DEAE` → `3_hash_`) using `str.extract(r"(.*_)")`
3. Convert Excel Serial Date floats → `pd.Timestamp`
4. Convert `Duration` (fractional days) → `Duration_hours`

Saves each cleaned table as parquet to `EVENTS_CLEANED_DIR`.

In [4]:
EXCEL_EPOCH = pd.Timestamp('1899-12-30')


def excel_serial_to_dt(series):
    """Convert a timestamp column to datetime64, handling three storage formats:
    1. datetime64 — pandas already parsed it from a clean Excel date cell → return as-is
    2. object dtype — contains Python datetime objects or datetime strings
       (happens when a column has mixed types or blank cells in the source Excel)
       → use pd.to_datetime() which handles both formats
    3. numeric — Excel serial float from CSV (e.g. 44622.576) → convert via epoch
    """
    if pd.api.types.is_datetime64_any_dtype(series):
        return series
    if series.dtype == object:
        return pd.to_datetime(series, errors='coerce')
    numeric = pd.to_numeric(series, errors='coerce')
    result = pd.Series(pd.NaT, index=series.index)
    is_unix = numeric > 1_000_000
    result[~is_unix] = EXCEL_EPOCH + pd.to_timedelta(numeric[~is_unix], unit='D')
    result[is_unix] = pd.to_datetime(numeric[is_unix], unit='ms', errors='coerce')
    return result


def safe_load_file(file_path, separator=','):
    """Scale-safe loader: chunked CSV or full Excel, respects DEV_MODE."""
    file_path = Path(file_path)
    ext = file_path.suffix.lower()
    if ext in ['.csv', '.txt']:
        if DEV_MODE and SAMPLE_ROWS is not None:
            return pd.read_csv(file_path, sep=separator, nrows=SAMPLE_ROWS, encoding='latin1')
        chunks = pd.read_csv(file_path, sep=separator, chunksize=CHUNKSIZE, encoding='latin1')
        return pd.concat(
            [chunk for chunk in tqdm(chunks, desc=f'Loading {file_path.name}')],
            ignore_index=True
        )
    elif ext in ['.xlsx', '.xls']:
        if DEV_MODE and SAMPLE_ROWS is not None:
            return pd.read_excel(file_path, nrows=SAMPLE_ROWS)
        print(f'Loading full Excel file: {file_path.name}...')
        return pd.read_excel(file_path)
    raise ValueError(f"Unsupported format '{ext}': {file_path.name}")


def clean_events_table(df, name=''):
    """Clean one Events table:
    1. Strip hidden nulls — but skip timestamp columns to avoid corrupting
       datetime objects that are stored as object dtype in the source Excel
    2. Forward-fill BatchID
    3. Normalize BatchID: strip stage suffix for production IDs, keep others unchanged
    4. Parse Start time / End time to datetime64
    5. Compute Duration_hours = End time - Start time
    """
    df = df.copy()

    TS_COLS = {'Start time', 'End time'}
    HIDDEN = {'No_Child_1/2', 'No sample available', '', ' '}

    # Do NOT process timestamp columns here: .str.strip() on an object Series
    # converts Python datetime objects to NaN, silently destroying the data
    for col in df.select_dtypes(include='object').columns:
        if col in TS_COLS:
            continue
        df[col] = df[col].str.strip().replace(list(HIDDEN), np.nan)

    # Forward-fill BatchID
    df['BatchID'] = df['BatchID'].ffill()

    # Normalize production batch IDs: '3_<hash>_DEAE' → '3_<hash>_'
    # Fall back to original for CIP/STOCK/HETP IDs that contain no underscore suffix
    normalized = df['BatchID'].str.extract(r'(.*_)')[0]
    df['BatchID'] = normalized.fillna(df['BatchID'])

    # Parse timestamps
    for ts_col in ['Start time', 'End time']:
        if ts_col in df.columns:
            df[ts_col] = excel_serial_to_dt(df[ts_col])

    # Compute Duration_hours from actual timestamps
    if 'Start time' in df.columns and 'End time' in df.columns:
        df['Duration_hours'] = (
            (df['End time'] - df['Start time']).dt.total_seconds() / 3600
        ).round(4)

    if 'Duration' in df.columns:
        df = df.drop(columns=['Duration'])

    return df


# Load 7 event files — UF-events_TRAITEMENT.xlsx excluded per spec
event_files = [
    f for f in sorted(EVENTS_DIR.glob('*.xlsx'))
    if "UF-events_TRAITEMENT" not in f.name
]
print(f'Event files to process: {len(event_files)}')

EVENTS_CLEANED_DIR.mkdir(parents=True, exist_ok=True)

for path in event_files:
    raw = safe_load_file(path)
    cleaned = clean_events_table(raw, path.stem)
    parquet_name = f'events_{path.stem.lower()}_cleaned.parquet'
    save_path = EVENTS_CLEANED_DIR / parquet_name
    cleaned.to_parquet(save_path, index=False)
    print(
        f'  {path.stem:<45s}  shape={cleaned.shape}  '
        f'BatchID_nulls={cleaned["BatchID"].isna().sum()}  '
        f'StartTime_nulls={cleaned["Start time"].isna().sum()}  '
        f'EndTime_nulls={cleaned["End time"].isna().sum()}  '
        f'-> {parquet_name}'
    )

print(f'\nAll {len(event_files)} event files cleaned and saved to {EVENTS_CLEANED_DIR}')

Event files to process: 7
Loading full Excel file: XTO_DEAE_events_UP12_batches.xlsx...
  XTO_DEAE_events_UP12_batches                   shape=(53076, 9)  BatchID_nulls=0  StartTime_nulls=0  EndTime_nulls=0  -> events_xto_deae_events_up12_batches_cleaned.parquet
Loading full Excel file: XTO_PG_DEAE_events_UP1.xlsx...
  XTO_PG_DEAE_events_UP1                         shape=(78786, 9)  BatchID_nulls=0  StartTime_nulls=0  EndTime_nulls=0  -> events_xto_pg_deae_events_up1_cleaned.parquet
Loading full Excel file: XTO_PG_DEAE_events_UP2.xlsx...
  XTO_PG_DEAE_events_UP2                         shape=(51820, 9)  BatchID_nulls=0  StartTime_nulls=0  EndTime_nulls=0  -> events_xto_pg_deae_events_up2_cleaned.parquet
Loading full Excel file: XTO_PG_DEAE_events_UP5.xlsx...
  XTO_PG_DEAE_events_UP5                         shape=(90595, 9)  BatchID_nulls=0  StartTime_nulls=0  EndTime_nulls=1  -> events_xto_pg_deae_events_up5_cleaned.parquet
Loading full Excel file: XTO_PG_events_UP3.xlsx...
  XTO_PG_ev

## Cell 5 — Events Aggregation (per batch per phase)

For each of the 8 phases, computes full-phase duration stats per `BatchID`:
count, total, mean, std, max of `Duration_hours` — prefixed as `ev_{phase}_{metric}`.

Elution detection is **not** done here because the Events `Event name` and `Child 1`
columns contain opaque operation codes, not human-readable step labels. Elution features
are extracted in Cell 7 from the sensor `UV Elution` boolean flag instead.

All phases are outer-merged into one wide `events_features_df`.

In [5]:
events_files = {
    'pg_deae_up1' : 'events_xto_pg_deae_events_up1_cleaned.parquet',
    'pg_deae_up2' : 'events_xto_pg_deae_events_up2_cleaned.parquet',
    'pg_up3'      : 'events_xto_pg_events_up3_cleaned.parquet',
    'pg_up4'      : 'events_xto_pg_events_up4_batches_cleaned.parquet',
    'pg_deae_up5' : 'events_xto_pg_deae_events_up5_cleaned.parquet',
    'pg_up6'      : 'events_xto_pg_events_up6_cleaned.parquet',
    'deae_up12'   : 'events_xto_deae_events_up12_batches_cleaned.parquet',
}

events_agg_list = []

for phase, fname in events_files.items():
    df = pd.read_parquet(EVENTS_CLEANED_DIR / fname)

    # Restrict to valid SAP batch IDs
    df = df[df['BatchID'].isin(VALID_BATCH_IDS)]
    if len(df) == 0:
        print(f'  {phase}: no valid batches after filtering — skipping')
        continue

    # Full-phase duration features
    agg = (
        df.groupby('BatchID', as_index=False)
        .agg(
            n_events         =('Event name',     'count'),
            total_duration_h =('Duration_hours', 'sum'),
            mean_duration_h  =('Duration_hours', 'mean'),
            std_duration_h   =('Duration_hours', 'std'),
            max_duration_h   =('Duration_hours', 'max'),
        )
    )
    rename_map = {c: f'ev_{phase}_{c}' for c in agg.columns if c != 'BatchID'}
    agg = agg.rename(columns=rename_map)

    events_agg_list.append(agg)
    print(f'  {phase:<15s}: {len(agg):3d} batches, {agg.shape[1]-1:2d} features')

if events_agg_list:
    events_features_df = reduce(
        lambda a, b: a.merge(b, on='BatchID', how='outer'),
        events_agg_list
    )
else:
    events_features_df = pd.DataFrame(columns=['BatchID'])
    print('WARNING: No events aggregations produced.')

print(f'\nevents_features_df shape: {events_features_df.shape}')
print(f'Unique BatchIDs: {events_features_df["BatchID"].nunique()}')

  pg_deae_up1    : 205 batches,  5 features
  pg_deae_up2: no valid batches after filtering — skipping
  pg_up3: no valid batches after filtering — skipping
  pg_up4         : 204 batches,  5 features
  pg_deae_up5    : 205 batches,  5 features
  pg_up6         :   9 batches,  5 features
  deae_up12      : 205 batches,  5 features

events_features_df shape: (207, 26)
Unique BatchIDs: 207


## Cell 5b — Child 1 / Child 2 Sub-Event Feature Extraction

Extracts **sub-event duration features** from the two focus event files identified by GSK Philippe:
- `pg_up4` (XTO_PG_events_UP4_batches) — PG chromatography production runs
- `deae_up12` (XTO_DEAE_events_UP12_batches) — DEAE chromatography production runs

**GSK rule:** Events are structured as Russian dolls — each top-level phase contains `Child 1`
subdivisions, which in turn contain `Child 2` subdivisions. Each unique `Child 2` value
represents an independent chromatography test/step and becomes its own feature column.

**Last-run-wins:** when the same child event appears multiple times for a batch (e.g., BIS/TER
re-runs), only the most recent occurrence is kept — consistent with the Cell 6 time-window logic.

**80% missing filter:** sub-event columns present in fewer than 20% of batches are dropped.
This removes rare or equipment-specific steps that would add noise rather than signal.

Output: `sub_events_wide` — one row per batch, columns named `{stage}_{level}_{event_name}`
(e.g. `pg_child2_chromatography_procedure`, `deae_child1_equilibration`).

In [6]:
FOCUS_PHASES = {
    'pg_up4'   : 'events_xto_pg_events_up4_batches_cleaned.parquet',
    'deae_up12': 'events_xto_deae_events_up12_batches_cleaned.parquet',
}

sub_event_dfs = []

for phase, fname in FOCUS_PHASES.items():
    df = pd.read_parquet(EVENTS_CLEANED_DIR / fname)
    df = df[df['BatchID'].isin(VALID_BATCH_IDS)].copy()
    stage_prefix = PHASE_TO_STAGE.get(phase, phase).lower()  # 'pg' or 'deae'

    for level, col in [('child1', 'Child 1'), ('child2', 'Child 2')]:
        if col not in df.columns:
            print(f'  {phase}: column {col!r} not found — skipping')
            continue

        # Drop rows where the child column is null
        sub = df.dropna(subset=[col]).copy()
        if len(sub) == 0:
            print(f'  {phase}/{col}: all null — skipping')
            continue

        # Clean child event name → use as column name
        sub['_event_key'] = (
            sub[col]
            .str.strip()
            .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)
            .str.lower()
        )

        # Pivot: one column per unique child event name, value = duration_hours
        # For each batch, if the same child event appears multiple times,
        # take the LAST one (GSK rule: last run wins)
        sub_sorted = sub.sort_values('Start time', ascending=True)
        pivot = (
            sub_sorted
            .groupby(['BatchID', '_event_key'])['Duration_hours']
            .last()
            .unstack(fill_value=np.nan)
        )
        pivot.columns = [f'{stage_prefix}_{level}_{c}' for c in pivot.columns]
        pivot = pivot.reset_index()

        # Drop columns with >80% missing values
        n_batches = pivot['BatchID'].nunique()
        missing_pct = pivot.drop(columns='BatchID').isna().mean()
        keep_cols = missing_pct[missing_pct <= 0.80].index.tolist()
        dropped = len(missing_pct) - len(keep_cols)
        pivot = pivot[['BatchID'] + keep_cols]

        print(f'  {phase}/{level}: {len(keep_cols)} columns kept '
              f'({dropped} dropped for >80% missing)')

        if len(keep_cols) > 0:
            sub_event_dfs.append(pivot)

# Merge all sub-event feature tables
if sub_event_dfs:
    sub_events_wide = reduce(
        lambda a, b: a.merge(b, on='BatchID', how='outer'),
        sub_event_dfs
    )
    print(f'\nsub_events_wide shape: {sub_events_wide.shape}')
    print(f'Unique BatchIDs: {sub_events_wide["BatchID"].nunique()}')
else:
    sub_events_wide = pd.DataFrame(columns=['BatchID'])
    print('WARNING: No sub-event features produced.')

  pg_up4/child1: 2 columns kept (1 dropped for >80% missing)
  pg_up4/child2: 162 columns kept (53 dropped for >80% missing)
  deae_up12/child1: 2 columns kept (1 dropped for >80% missing)
  deae_up12/child2: 183 columns kept (8 dropped for >80% missing)

sub_events_wide shape: (207, 350)
Unique BatchIDs: 207


## Cell 5c — Child 2 Sub-Event XTO Time Window Map

Unlike Cell 6 which uses top-level unit procedure rows (Child 1 = NaN,
full batch window), this cell uses Phase-level rows (Child 2 populated)
to build narrower, sub-step-specific time windows for sensor matching.

For each surviving Child 2 sub-step type, sensor readings will be
extracted only from the time window when that specific sub-step was running.

In [7]:
CHILD2_NOISE_PATTERNS = [
    'alarm', 'prompt', 'gen_get_val', 'user_request', 'cip','init',
]

CHILD2_MIN_DURATION_H   = 30 / 60   # 30 minutes — filters system handshakes
CHILD2_MIN_BATCH_COVERAGE = 0.20   # sub-step must appear in >=20% of batches

FOCUS_PHASES_MAP = {
    'pg_up4'   : ('events_xto_pg_events_up4_batches_cleaned.parquet',   'PG'),
    'deae_up12': ('events_xto_deae_events_up12_batches_cleaned.parquet', 'DEAE'),
}

child2_map_rows = []

for phase, (fname, stage) in FOCUS_PHASES_MAP.items():
    df = pd.read_parquet(EVENTS_CLEANED_DIR / fname)
    df = df[df['BatchID'].isin(VALID_BATCH_IDS)].copy()

    # Phase-level rows have BOTH Primary element = XTO AND Child 2 populated
    child2_mask = (
        df['Primary element'].astype(str).str.match(r'^XTO\d+$', na=False) &
        df['Child 2'].notna()
    )
    sub = df[child2_mask].copy()

    if len(sub) == 0:
        print(f'  {phase}: no Phase-level XTO rows found — skipping')
        continue

    # Clean Child 2 name into a step key:
    #   1. Replace non-alphanumeric chars with _
    #   2. Strip leading/trailing underscores
    #   3. Lowercase
    sub['_step_key'] = (
        sub['Child 2']
        .str.strip()
        .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)
        .str.strip('_')
        .str.lower()
    )

    # Noise filter
    noise_mask = sub['_step_key'].apply(
        lambda k: any(p in k for p in CHILD2_NOISE_PATTERNS)
    )
    n_before = len(sub)
    sub = sub[~noise_mask]
    print(f'  {phase}: {n_before} rows \u2192 {len(sub)} after noise filter '
          f'({noise_mask.sum()} removed)')

    # Duration filter
    sub = sub[sub['Duration_hours'] >= CHILD2_MIN_DURATION_H]
    print(f'  {phase}: {len(sub)} rows after duration filter (\u226530 min)')

    # Last-run-wins at sub-step level
    sub = (
        sub.sort_values('Start time', ascending=True)
        .groupby(['BatchID', '_step_key'], as_index=False)
        .last()
    )

    # Coverage filter: keep step types present in >= CHILD2_MIN_BATCH_COVERAGE of batches
    min_batches = int(np.ceil(CHILD2_MIN_BATCH_COVERAGE * len(VALID_BATCH_IDS)))
    step_counts = sub.groupby('_step_key')['BatchID'].nunique()
    valid_steps = step_counts[step_counts >= min_batches].index
    sub = sub[sub['_step_key'].isin(valid_steps)]
    print(f'  {phase}: {len(valid_steps)} sub-step types survive coverage filter '
          f'(each in \u2265{min_batches} batches)')
    print(f'  Surviving steps: {sorted(valid_steps.tolist())}')

    # Build XTO map rows for this phase
    rows = sub[['Primary element', 'BatchID', 'Start time', 'End time', '_step_key']].copy()
    rows = rows.rename(columns={'Primary element': 'XTO', '_step_key': 'sub_step'})
    rows['stage'] = stage
    child2_map_rows.append(rows)

if child2_map_rows:
    child2_xto_map = pd.concat(child2_map_rows, ignore_index=True)
    child2_xto_map = child2_xto_map.dropna(
        subset=['XTO', 'BatchID', 'Start time', 'End time']
    )
    print(f'\nchild2_xto_map shape: {child2_xto_map.shape}')
    print(f'Unique sub-step types: {child2_xto_map["sub_step"].nunique()}')
    print(f'Unique BatchIDs:       {child2_xto_map["BatchID"].nunique()}')
    print(f'Stage breakdown:')
    print(child2_xto_map.groupby(['stage', 'sub_step'])['BatchID'].nunique()
          .rename('n_batches').to_string())
else:
    child2_xto_map = pd.DataFrame(
        columns=['XTO', 'BatchID', 'Start time', 'End time', 'sub_step', 'stage']
    )
    print('WARNING: child2_xto_map is empty — no qualifying sub-steps found.')

  pg_up4: 41194 rows → 32182 after noise filter (9012 removed)
  pg_up4: 8111 rows after duration filter (≥30 min)
  pg_up4: 44 sub-step types survive coverage filter (each in ≥42 batches)
  Surviving steps: ['xto_common_20x__28_1', 'xto_common_20x__29_1', 'xto_common_20x__30_1', 'xto_common_20x__32_1', 'xto_common_20x__34_1', 'xto_common_20x__35_1', 'xto_common_20x__36_1', 'xto_common_20x__37_1', 'xto_common_20x__3_1', 'xto_common_20x__41_1', 'xto_common_20x__43_1', 'xto_common_20x__44_1', 'xto_control_20x__14_1', 'xto_control_20x__15_1', 'xto_control_20x__19_1', 'xto_control_20x__1_1', 'xto_control_20x__20_1', 'xto_control_20x__21_1', 'xto_control_20x__23_1', 'xto_control_20x__24_1', 'xto_control_20x__28_1', 'xto_control_20x__30_1', 'xto_control_20x__4_1', 'xto_control_20x__7_1', 'xto_inlet_20x__13_1', 'xto_inlet_20x__1_1', 'xto_inlet_20x__4_1', 'xto_inlet_20x__5_1', 'xto_inlet_20x__6_1', 'xto_inlet_20x__7_1', 'xto_inline_201__11_1', 'xto_inline_201__16_1', 'xto_inline_201__1_1', 'xt

## Cell 5d — Sub-Event Sensor Feature Extraction (DuckDB range join)

Runs a second DuckDB range join pass using child2_xto_map (sub-step
time windows) instead of the top-level batch windows from Cell 6.

Produces sensor features scoped to each specific Child 2 sub-step,
e.g. `pg_sub_xto_inlet_20x_sensor_UV_Light_mean` captures the UV
reading specifically during the XTO_INLET_20X phase of PG.

Column naming convention: `{stage_lower}_sub_{step_name}_sensor_{col_name}_{agg_fn}`

In [8]:
# Guard: if child2_xto_map is empty, skip processing
if child2_xto_map.empty:
    child2_sensor_wide = pd.DataFrame(columns=['ev_batch_id'])
    print('child2_xto_map is empty — skipping sub-event sensor extraction.')
else:
    all_child2_aggs = []
    c2_total_rows    = 0
    c2_total_matched = 0

    # Discover sensor files independently (Cell 5d runs before Cell 7)
    child2_sensor_files = sorted(
        f for f in SENSOR_DIR.glob('*.*')
        if f.name.startswith('IPV_IESG') or f.name.startswith('XTO')
    )
    print(f'Sensor files for child2 pass: {len(child2_sensor_files)}')

    for f in child2_sensor_files:
        print(f'\nProcessing {f.name}...')

        # ── Step 0: Identify XTO of this file with a 3-row probe ──────────
        # This avoids reading the full file if no child2 windows exist for it.
        probe = pd.read_csv(
            f, sep='\t', encoding='latin1', nrows=3,
            engine='python', on_bad_lines='skip', quoting=csv.QUOTE_NONE
        )
        probe_col = probe.columns[0]
        file_xto  = str(probe[probe_col].iloc[0]).strip() if len(probe) > 0 else None

        c2_map_filt = child2_xto_map[child2_xto_map['XTO'] == file_xto].copy()

        if c2_map_filt.empty:
            print(f'  No child2 windows for XTO={file_xto} — skipping file.')
            continue

        # Determine time bounds and valid XTOs from the map BEFORE reading
        t_min      = c2_map_filt['Start time'].min()
        t_max      = c2_map_filt['End time'].max()
        valid_xtos = set(c2_map_filt['XTO'].unique())

        print(f'  XTO={file_xto} | child2 windows: {len(c2_map_filt)} | '
              f'time range: {t_min} → {t_max}')

        # ── Step 1: Read with IN-LOOP filtering ───────────────────────────
        # Filter each 100K-row chunk immediately — never accumulate the
        # full 35M rows. Only rows matching the XTO and time window are kept.
        chunks = pd.read_csv(
            f, sep='\t', encoding='latin1', chunksize=CHUNKSIZE,
            engine='python', on_bad_lines='skip', quoting=csv.QUOTE_NONE
        )

        processed_chunks = []
        rows_read     = 0
        rows_kept     = 0

        for chunk in tqdm(chunks, desc='Reading', leave=False):
            rows_read += len(chunk)

            # Rename first column to XTO if needed
            if chunk.columns[0] != 'XTO':
                chunk = chunk.rename(columns={chunk.columns[0]: 'XTO'})

            # Parse timestamp
            chunk['_timestamp'] = pd.to_datetime(
                chunk['GMT TimeStamp'], dayfirst=True, errors='coerce'
            )

            # ★ EARLY FILTER — discard non-matching rows immediately ★
            mask = (
                chunk['XTO'].isin(valid_xtos) &
                (chunk['_timestamp'] >= t_min) &
                (chunk['_timestamp'] <= t_max)
            )
            chunk = chunk[mask]

            if len(chunk) == 0:
                continue   # nothing in this chunk — skip

            # Type conversions only on the small filtered subset
            for c in SENSOR_NUMERIC_COLS:
                if c in chunk.columns:
                    chunk[c] = pd.to_numeric(chunk[c], errors='coerce').astype('float32')
            for c in SENSOR_BOOL_COLS:
                if c in chunk.columns:
                    chunk[c] = (
                        chunk[c].map({True: 1, False: 0, 'True': 1, 'False': 0})
                        .astype('float32')
                    )

            rows_kept += len(chunk)
            processed_chunks.append(chunk)

        print(f'  Rows read: {rows_read:,} | Rows kept after filter: {rows_kept:,} '
              f'({rows_kept/rows_read*100:.1f}%)')

        if not processed_chunks:
            print(f'  No matching rows after filtering — skipping file.')
            del processed_chunks
            gc.collect()
            continue

        sensor_tagged = pd.concat(processed_chunks, ignore_index=True)
        del processed_chunks
        gc.collect()
        print(f'  sensor_tagged shape: {sensor_tagged.shape}')

        # ── Step 2: DuckDB range join ─────────────────────────────────────
        matched = duckdb.sql('''
            SELECT
                s.*,
                x.BatchID  AS _ev_batch,
                x.sub_step AS _ev_substep,
                x.stage    AS _ev_stage
            FROM sensor_tagged AS s
            INNER JOIN c2_map_filt AS x
                ON  s.XTO        =  x.XTO
                AND s._timestamp >= x."Start time"
                AND s._timestamp <= x."End time"
        ''').df()

        c2_total_rows    += len(sensor_tagged)
        c2_total_matched += len(matched)
        print(f'  Matched {len(matched):,} rows from {len(sensor_tagged):,}')

        avail_cols = [c for c in SENSOR_NUMERIC_COLS + SENSOR_BOOL_COLS
                      if c in matched.columns]

        if len(matched) > 0:
            # Aggregate by (batch, sub_step) — NOT by stage alone
            c2_agg = (
                matched
                .groupby(['_ev_batch', '_ev_substep', '_ev_stage'])[avail_cols]
                .agg(['mean', 'std', 'min', 'max'])
            )
            c2_agg.columns = [f'sensor_{c}_{fn}' for c, fn in c2_agg.columns]
            c2_agg = c2_agg.reset_index()
            all_child2_aggs.append(c2_agg)

        del sensor_tagged, matched
        gc.collect()

    print(f'\nTotal sensor rows processed (child2 pass): {c2_total_rows:,}')
    print(f'Total rows matched:                        {c2_total_matched:,}')

    if all_child2_aggs:
        child2_long = pd.concat(all_child2_aggs, ignore_index=True)

        # Re-aggregate across files (batch split across daily files)
        numeric_cols = child2_long.select_dtypes(include='number').columns.tolist()
        child2_long = (
            child2_long
            .groupby(['_ev_batch', '_ev_substep', '_ev_stage'], as_index=False)
            [numeric_cols]
            .mean()
        )

        # Build column prefix: {stage_lower}_sub_{step_name}
        child2_long['_col_prefix'] = (
            child2_long['_ev_stage'].str.lower() + '_sub_' +
            child2_long['_ev_substep']
        )

        # Pivot to wide: one row per batch
        child2_wide_parts = []
        for prefix, grp in child2_long.groupby('_col_prefix'):
            sensor_cols_grp = [c for c in grp.columns if c.startswith('sensor_')]
            grp_wide = (
                grp[['_ev_batch'] + sensor_cols_grp]
                .rename(columns={'_ev_batch': 'ev_batch_id'})
                .rename(columns={c: f'{prefix}_{c}' for c in sensor_cols_grp})
            )
            child2_wide_parts.append(grp_wide)

        if child2_wide_parts:
            child2_sensor_wide = reduce(
                lambda a, b: a.merge(b, on='ev_batch_id', how='outer'),
                child2_wide_parts
            )
        else:
            child2_sensor_wide = pd.DataFrame(columns=['ev_batch_id'])

        print(f'\nchild2_sensor_wide shape: {child2_sensor_wide.shape}')
        print(f'Unique batches:     {child2_sensor_wide["ev_batch_id"].nunique()}')
        print(f'Sub-step groups:    {len(child2_wide_parts)}')
        print(f'Sample columns:     {child2_sensor_wide.columns[1:4].tolist()}')
    else:
        child2_sensor_wide = pd.DataFrame(columns=['ev_batch_id'])
        print('WARNING: No child2 sensor features produced.')


Sensor files for child2 pass: 7

Processing IPV_IESG_XTO53101_20260219104626.txt...
  XTO=XTO53101 | child2 windows: 1057 | time range: 2022-03-21 17:37:47.965000 → 2023-06-08 14:41:46.172000


  Rows read: 13,047,405 | Rows kept after filter: 3,835,104 (29.4%)
  sensor_tagged shape: (3835104, 60)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Matched 3,534,536 rows from 3,835,104

Processing IPV_IESG_XTO53103_20260219104651.txt...
  XTO=XTO53103 | child2 windows: 2879 | time range: 2022-02-28 18:49:05.290000 → 2025-12-09 08:56:34.094000


  Rows read: 35,732,978 | Rows kept after filter: 832,033 (2.3%)
  sensor_tagged shape: (832033, 60)
  Matched 1,471,081 rows from 832,033

Processing IPV_IESG_XTO53104_20260219104707.txt...
  XTO=XTO53104 | child2 windows: 2682 | time range: 2022-03-07 18:00:24.698000 → 2025-09-23 10:03:54.153000


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Matched 1,644,877 rows from 3,818,646

Processing IPV_IESG_XTO53601_20260219104808.txt...
  XTO=XTO53601 | child2 windows: 3417 | time range: 2022-03-02 14:03:41.693000 → 2025-12-08 11:00:36.757000


  Rows read: 13,047,415 | Rows kept after filter: 11,896,181 (91.2%)
  sensor_tagged shape: (11896181, 60)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Matched 6,024,081 rows from 11,896,181

Total sensor rows processed (child2 pass): 44,102,176
Total rows matched:                        27,531,770

child2_sensor_wide shape: (206, 6249)
Unique batches:     206
Sub-step groups:    71
Sample columns:     ['deae_sub_xto_common_20x__29_1_sensor_AI_A1_mean', 'deae_sub_xto_common_20x__29_1_sensor_AI_A1_std', 'deae_sub_xto_common_20x__29_1_sensor_AI_A1_min']


## Cell 6 — Build XTO-to-Batch Event Window Map

Scans all 7 Events parquet files for rows where `Primary element` matches `^XTO\d+$`
**and `Child 1` is NaN** (top-level `Chromatography_UnitProcedure` rows only).

Each batch has three hierarchy levels sharing the same `Primary element`:
- **UnitProcedure** (Child 1 = NaN) — full batch window, e.g. 23 hours ← **we keep only this**
- **Operation** (Child 1 populated, Child 2 = NaN) — sub-step, e.g. 4 hours
- **Phase** (Child 1 + Child 2 populated) — sub-phase, e.g. 10 seconds

Without this filter, `groupby.last()` would select the latest-starting row — often a
10-second Phase sub-row near the end of the batch — collapsing the sensor window to
seconds and matching almost no data.

After filtering to UnitProcedure level, two additional de-duplication steps run:
1. **Last-run-wins** — keeps the most recent time window per (BatchID, phase), discarding earlier failed runs (_PGBIS, _PGBIS1, etc.)
2. **DEAE_UP12 preference** — removes `pg_deae_up1` entries that duplicate a `deae_up12` time window.

In [9]:
xto_map_rows = []

for phase, fname in events_files.items():
    ev = pd.read_parquet(EVENTS_CLEANED_DIR / fname)

    # Child 1 is NaN only on top-level Chromatography_UnitProcedure rows,
    # which carry the full batch time window (e.g. 23 hours).
    # Filtering here prevents Operation and Phase sub-rows (Child 1/2 populated)
    # from polluting the map with tiny time windows (down to 10 seconds).
    mask = (
        ev['Primary element'].astype(str).str.match(r'^XTO\d+$', na=False) &
        ev['Child 1'].isna()
    )
    ev_xto = ev.loc[mask, ['Primary element', 'BatchID', 'Start time', 'End time']].copy()
    ev_xto['phase'] = phase
    ev_xto['stage'] = PHASE_TO_STAGE.get(phase, 'PREP')
    if len(ev_xto):
        xto_map_rows.append(ev_xto)

if xto_map_rows:
    xto_map = (
        pd.concat(xto_map_rows, ignore_index=True)
        .rename(columns={'Primary element': 'XTO'})
        .dropna(subset=['XTO', 'BatchID', 'Start time', 'End time'])
    )

    # CRITICAL FILTER: reduces ~204,117 rows to ~24,000 rows covering valid SAP batches
    xto_map = xto_map[xto_map['BatchID'].isin(VALID_BATCH_IDS)]

    xto_map = xto_map.drop_duplicates(subset=['XTO', 'BatchID', 'Start time'])

    # LAST-RUN-WINS: for each (BatchID, phase), keep only the window
    # with the latest Start time — discards earlier failed runs (BIS/TER naming)
    xto_map = (
        xto_map
        .sort_values('Start time', ascending=True)
        .groupby(['BatchID', 'phase'], as_index=False)
        .last()
    )
    print(f'After last-run-wins filter:')
    print(f'  Total entries:   {len(xto_map):,}')
    print(f'  Unique BatchIDs: {xto_map["BatchID"].nunique()}')
    print(xto_map['stage'].value_counts().to_string())

    # DEAE_UP12 PREFERENCE: when a batch has the same time window in both
    # deae_up12 and pg_deae_up1, keep only the deae_up12 entry.
    deae_windows = xto_map[xto_map['phase'] == 'deae_up12'].set_index(
        ['BatchID', 'Start time', 'End time']
    ).index

    dup_mask = (
        (xto_map['phase'] == 'pg_deae_up1') &
        xto_map.set_index(['BatchID', 'Start time', 'End time']).index.isin(deae_windows)
    )
    n_removed = dup_mask.sum()
    xto_map = xto_map[~dup_mask].reset_index(drop=True)
    print(f'Removed {n_removed} PG_DEAE_UP1 rows that were duplicates of DEAE_UP12 entries')

    print(f'\nUnique XTO IDs:  {xto_map["XTO"].nunique()}')
    print(f'Total entries:   {len(xto_map):,}')
    print(f'Unique BatchIDs: {xto_map["BatchID"].nunique()}')
    print(f'Stage counts:')
    print(xto_map['stage'].value_counts().to_string())
else:
    xto_map = pd.DataFrame(
        columns=['XTO', 'BatchID', 'Start time', 'End time', 'phase', 'stage']
    )
    print('WARNING: No XTO->Batch mapping found in events.')

After last-run-wins filter:
  Total entries:   828
  Unique BatchIDs: 207
stage
PREP    419
DEAE    205
PG      204
Removed 0 PG_DEAE_UP1 rows that were duplicates of DEAE_UP12 entries

Unique XTO IDs:  7
Total entries:   828
Unique BatchIDs: 207
Stage counts:
stage
PREP    419
DEAE    205
PG      204


## Cell 7 — Sensor Processing Pipeline (DuckDB range join)

Processes each sensor `.txt` file in four phases:
1. **Load** in chunks (chunked CSV, memory-safe)
2. **Pre-filter** sensor rows to the valid time window
3. **DuckDB range join** — matches each sensor row to a batch by XTO + timestamp overlap
4. **Aggregate** matched rows to (batch, stage)-level statistics; free RAM immediately

The DuckDB join replaces the O(n_events × n_sensor_rows) `iterrows()` loop from v1.

In [10]:
# Find sensor files
sensor_files = sorted(
    f for f in SENSOR_DIR.glob('*.*')
    if f.name.startswith('IPV_IESG') or f.name.startswith('XTO')
)
print(f'Sensor files found: {len(sensor_files)}')
for f in sensor_files:
    print(f'  {f.name}')

all_file_aggs = []
total_assigned = 0
total_rows = 0

for f in sensor_files:
    print('=' * 60)
    print(f'PHASE 1: Loading {f.name}')

    # Load sensor file in chunks
    chunks = pd.read_csv(
        f, sep='\t', encoding='latin1', chunksize=CHUNKSIZE,
        engine='python', on_bad_lines='skip', quoting=csv.QUOTE_NONE
    )

    processed_chunks = []
    for chunk in tqdm(chunks, desc='Reading'):
        if chunk.columns[0] != 'XTO':
            chunk = chunk.rename(columns={chunk.columns[0]: 'XTO'})
        chunk['_timestamp'] = pd.to_datetime(
            chunk['GMT TimeStamp'], dayfirst=True, errors='coerce'
        )
        for c in SENSOR_NUMERIC_COLS:
            if c in chunk.columns:
                chunk[c] = pd.to_numeric(chunk[c], errors='coerce').astype('float32')
        for c in SENSOR_BOOL_COLS:
            if c in chunk.columns:
                chunk[c] = (
                    chunk[c].map({True: 1, False: 0, 'True': 1, 'False': 0})
                    .astype('float32')
                )
        processed_chunks.append(chunk)

    sensor_tagged = pd.concat(processed_chunks, ignore_index=True)
    del processed_chunks
    gc.collect()
    print(f'  Loaded {len(sensor_tagged):,} rows')

    # Pre-filter: keep only rows within the valid batch time range
    current_xtos = set(sensor_tagged['XTO'].dropna().unique())
    xto_map_filt = xto_map[xto_map['XTO'].isin(current_xtos)].copy()

    if xto_map_filt.empty:
        print('  No matching XTO IDs — skipping file.')
        del sensor_tagged
        gc.collect()
        continue

    t_min = xto_map_filt['Start time'].min()
    t_max = xto_map_filt['End time'].max()
    sensor_tagged = sensor_tagged[
        (sensor_tagged['_timestamp'] >= t_min) &
        (sensor_tagged['_timestamp'] <= t_max)
    ].copy()
    print(f'  Sensor rows after time pre-filter: {len(sensor_tagged):,}')

    # Phase 2: DuckDB range join (replaces iterrows loop)
    print('  PHASE 2: DuckDB range join...')

    matched = duckdb.sql("""
        SELECT
            s.*,
            x.BatchID  AS _ev_batch,
            x.phase    AS _ev_phase,
            x.stage    AS _ev_stage
        FROM sensor_tagged   AS s
        INNER JOIN xto_map_filt AS x
            ON  s.XTO         =  x.XTO
            AND s._timestamp  >= x."Start time"
            AND s._timestamp  <= x."End time"
    """).df()

    print(f'  Matched {len(matched):,} rows from {len(sensor_tagged):,}')
    total_rows     += len(sensor_tagged)
    total_assigned += len(matched)

    # Phase 3: Aggregate matched rows
    avail_cols = [
        c for c in SENSOR_NUMERIC_COLS + SENSOR_BOOL_COLS
        if c in matched.columns
    ]

    if len(matched) > 0:
        # A) Full-run features (all matched rows)
        fullrun_agg = (
            matched
            .groupby(['_ev_batch', '_ev_stage'])[avail_cols]
            .agg(['mean', 'std', 'min', 'max'])
        )
        fullrun_agg.columns = [f'sensor_{c}_{fn}' for c, fn in fullrun_agg.columns]
        fullrun_agg = fullrun_agg.reset_index()

        # B) Elution-only features (UV Elution == 1)
        if 'UV Elution' in matched.columns:
            elution_rows = matched[matched['UV Elution'] == 1]
        else:
            elution_rows = pd.DataFrame()

        if len(elution_rows) > 0:
            elution_agg = (
                elution_rows
                .groupby(['_ev_batch', '_ev_stage'])[avail_cols]
                .agg(['mean', 'std', 'min', 'max'])
            )
            elution_agg.columns = [f'elution_{c}_{fn}' for c, fn in elution_agg.columns]
            elution_agg = elution_agg.reset_index()
            file_agg = fullrun_agg.merge(
                elution_agg, on=['_ev_batch', '_ev_stage'], how='left'
            )
        else:
            print('  WARNING: No elution rows found in this file.')
            file_agg = fullrun_agg

        all_file_aggs.append(file_agg)

    # Phase 4: Free RAM immediately
    del sensor_tagged, matched
    gc.collect()

print('=' * 60)
if total_rows > 0:
    print(f'Total rows processed: {total_rows:,}')
    print(f'Total rows matched:   {total_assigned:,}')
    print(f'Match rate:           {total_assigned / total_rows * 100:.1f}%')
else:
    print('No rows were processed.')

Sensor files found: 7
  IPV_IESG_XTO53101_20260219104626.txt
  IPV_IESG_XTO53103_20260219104651.txt
  IPV_IESG_XTO53104_20260219104707.txt
  IPV_IESG_XTO53201_20260219104720.txt
  IPV_IESG_XTO53202_20260219104737.txt
  IPV_IESG_XTO53501_20260219104754.txt
  IPV_IESG_XTO53601_20260219104808.txt
PHASE 1: Loading IPV_IESG_XTO53101_20260219104626.txt


Reading: 131it [07:40,  3.51s/it]


  Loaded 13,047,405 rows
  Sensor rows after time pre-filter: 3,835,188
  PHASE 2: DuckDB range join...
  Matched 718,656 rows from 3,835,188
PHASE 1: Loading IPV_IESG_XTO53103_20260219104651.txt


Reading: 360it [13:51,  2.31s/it]


  Loaded 35,732,978 rows
  Sensor rows after time pre-filter: 831,633
  PHASE 2: DuckDB range join...
  Matched 297,510 rows from 831,633
PHASE 1: Loading IPV_IESG_XTO53104_20260219104707.txt


...

  Loaded 13,047,414 rows
  Sensor rows after time pre-filter: 3,818,739
  PHASE 2: DuckDB range join...
  Matched 713,366 rows from 3,818,739
PHASE 1: Loading IPV_IESG_XTO53601_20260219104808.txt


Reading: 131it [08:12,  3.76s/it]


  Loaded 13,047,415 rows
  Sensor rows after time pre-filter: 11,896,266
  PHASE 2: DuckDB range join...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Matched 2,524,332 rows from 11,896,266
Total rows processed: 44,102,249
Total rows matched:   7,536,494
Match rate:           17.1%


## Cell 8 — Combine All File Aggregations

Concatenates per-file aggregations, re-groups to handle batches split across
multiple daily files, then pivots to a wide table with separate column groups
for PG, DEAE, and PREP stages (one row per batch).

In [11]:
if all_file_aggs:
    sensor_agg = pd.concat(all_file_aggs, ignore_index=True)

    # Re-aggregate: if the same batch appears in multiple files, take the mean
    numeric_cols = sensor_agg.select_dtypes(include='number').columns.tolist()
    sensor_agg = (
        sensor_agg
        .groupby(['_ev_batch', '_ev_stage'], as_index=False)[numeric_cols]
        .mean()
    )

    sensor_agg = sensor_agg.rename(columns={'_ev_batch': 'ev_batch_id'})

    print(f'Final sensor_agg shape: {sensor_agg.shape}')
    print(f'Unique batches: {sensor_agg["ev_batch_id"].nunique()}')
    print(f'Stages: {sensor_agg["_ev_stage"].value_counts().to_dict()}')

    # Split by stage
    pg_features   = sensor_agg[sensor_agg['_ev_stage'] == 'PG'].copy()
    deae_features = sensor_agg[sensor_agg['_ev_stage'] == 'DEAE'].copy()
    prep_features = sensor_agg[sensor_agg['_ev_stage'] == 'PREP'].copy()

    print(f'PG batches:   {len(pg_features)}')
    print(f'DEAE batches: {len(deae_features)}')
    print(f'PREP batches: {len(prep_features)}')

    def prefix_cols(df, prefix, key='ev_batch_id'):
        """Prefix all feature columns (not the key or stage) with the stage name."""
        feature_cols = [c for c in df.columns if c not in [key, '_ev_stage']]
        return df[[key] + feature_cols].rename(
            columns={c: f'{prefix}_{c}' for c in feature_cols}
        )

    pg_wide   = prefix_cols(pg_features,   'pg')
    deae_wide = prefix_cols(deae_features, 'deae')
    prep_wide = prefix_cols(prep_features, 'prep')

    sensor_wide = reduce(
        lambda a, b: a.merge(b, on='ev_batch_id', how='outer'),
        [pg_wide, deae_wide, prep_wide]
    )

    print(f'\nWide sensor table: {sensor_wide.shape}')
    print(f'Sample columns (first 10): {sensor_wide.columns[:10].tolist()}')
else:
    sensor_wide = pd.DataFrame(columns=['ev_batch_id'])
    print('WARNING: No sensor aggregations produced — sensor_wide is empty.')

Final sensor_agg shape: (550, 178)
Unique batches: 206
Stages: {'DEAE': 205, 'PREP': 202, 'PG': 143}
PG batches:   143
DEAE batches: 205
PREP batches: 202

Wide sensor table: (206, 529)
Sample columns (first 10): ['ev_batch_id', 'pg_sensor_AI_A1_mean', 'pg_sensor_AI_A1_std', 'pg_sensor_AI_A1_min', 'pg_sensor_AI_A1_max', 'pg_sensor_AI_A2_mean', 'pg_sensor_AI_A2_std', 'pg_sensor_AI_A2_min', 'pg_sensor_AI_A2_max', 'pg_sensor_AI_A3_mean']


## Cell 9 — Add is_elution_available Flag

Adds binary flags indicating whether elution sensor data was captured for each batch
at the PG and DEAE stages. Used downstream to filter or impute elution features.

In [12]:
pg_elution_col   = 'pg_elution_UV Light_mean'
deae_elution_col = 'deae_elution_UV Light_mean'

if pg_elution_col in sensor_wide.columns:
    sensor_wide['pg_has_elution_data'] = sensor_wide[pg_elution_col].notna().astype(int)
else:
    sensor_wide['pg_has_elution_data'] = 0
    print(f"WARNING: '{pg_elution_col}' not found — pg_has_elution_data set to 0")

if deae_elution_col in sensor_wide.columns:
    sensor_wide['deae_has_elution_data'] = sensor_wide[deae_elution_col].notna().astype(int)
else:
    sensor_wide['deae_has_elution_data'] = 0
    print(f"WARNING: '{deae_elution_col}' not found — deae_has_elution_data set to 0")

print(f'Batches with PG elution data:   {sensor_wide["pg_has_elution_data"].sum()}')
print(f'Batches with DEAE elution data: {sensor_wide["deae_has_elution_data"].sum()}')

Batches with PG elution data:   130
Batches with DEAE elution data: 194


## Cell 10 — Merge Events Features into Sensor Wide Table

Left-joins the events aggregation features (Cell 5) into the sensor wide table.
Result is one row per batch with sensor stats AND events duration stats.

In [13]:
full_sensor_features = (
    sensor_wide
    .merge(
        events_features_df.rename(columns={'BatchID': 'ev_batch_id'}),
        on='ev_batch_id', how='left'
    )
    .merge(
        sub_events_wide.rename(columns={'BatchID': 'ev_batch_id'}),
        on='ev_batch_id', how='left'
    )
    .merge(
        child2_sensor_wide,
        on='ev_batch_id', how='left'
    )
)
print(f'Full sensor + events + sub-events + child2 sensor shape: {full_sensor_features.shape}')
print(f'Total columns:  {full_sensor_features.shape[1]}')
print(f'Unique batches: {full_sensor_features["ev_batch_id"].nunique()}')
print(f'Sample columns (last 10): {full_sensor_features.columns[-10:].tolist()}')

Full sensor + events + sub-events + child2 sensor shape: (206, 7153)
Total columns:  7153
Unique batches: 206
Sample columns (last 10): ['pg_sub_xto_outlet_20x__9_1_sensor_RunCalc_UV_DuringHarvest_min', 'pg_sub_xto_outlet_20x__9_1_sensor_RunCalc_UV_DuringHarvest_max', 'pg_sub_xto_outlet_20x__9_1_sensor_RunCalc_UV_ForIntegral_v2_mean', 'pg_sub_xto_outlet_20x__9_1_sensor_RunCalc_UV_ForIntegral_v2_std', 'pg_sub_xto_outlet_20x__9_1_sensor_RunCalc_UV_ForIntegral_v2_min', 'pg_sub_xto_outlet_20x__9_1_sensor_RunCalc_UV_ForIntegral_v2_max', 'pg_sub_xto_outlet_20x__9_1_sensor_UV Elution_mean', 'pg_sub_xto_outlet_20x__9_1_sensor_UV Elution_std', 'pg_sub_xto_outlet_20x__9_1_sensor_UV Elution_min', 'pg_sub_xto_outlet_20x__9_1_sensor_UV Elution_max']


## Cell 11 — Save Sensor Features Parquet

Saves the final feature table to `OUTPUT_DIR/sensor_aggregated_features_v2.parquet`.
This file is the input for the modeling notebook.

In [14]:
save_path = OUTPUT_DIR / 'sensor_aggregated_features_v5.parquet'
full_sensor_features.to_parquet(save_path, index=False)
print(f'Saved to: {save_path}')
print(f'Shape: {full_sensor_features.shape}')
print(f'Columns: {full_sensor_features.columns.tolist()}')

Saved to: C:\Hackathon-GSK\data\processed\sensor_aggregated_features_v5.parquet
Shape: (206, 7153)
Columns: ['ev_batch_id', 'pg_sensor_AI_A1_mean', 'pg_sensor_AI_A1_std', 'pg_sensor_AI_A1_min', 'pg_sensor_AI_A1_max', 'pg_sensor_AI_A2_mean', 'pg_sensor_AI_A2_std', 'pg_sensor_AI_A2_min', 'pg_sensor_AI_A2_max', 'pg_sensor_AI_A3_mean', 'pg_sensor_AI_A3_std', 'pg_sensor_AI_A3_min', 'pg_sensor_AI_A3_max', 'pg_sensor_Accumulated Volume_mean', 'pg_sensor_Accumulated Volume_std', 'pg_sensor_Accumulated Volume_min', 'pg_sensor_Accumulated Volume_max', 'pg_sensor_FIC_C11_mean', 'pg_sensor_FIC_C11_std', 'pg_sensor_FIC_C11_min', 'pg_sensor_FIC_C11_max', 'pg_sensor_MT_I1_OUT_mean', 'pg_sensor_MT_I1_OUT_std', 'pg_sensor_MT_I1_OUT_min', 'pg_sensor_MT_I1_OUT_max', 'pg_sensor_MT_I2_OUT_mean', 'pg_sensor_MT_I2_OUT_std', 'pg_sensor_MT_I2_OUT_min', 'pg_sensor_MT_I2_OUT_max', 'pg_sensor_PIC_A1_mean', 'pg_sensor_PIC_A1_std', 'pg_sensor_PIC_A1_min', 'pg_sensor_PIC_A1_max', 'pg_sensor_PIC_I1_mean', 'pg_sensor_

## Cell 12 — Split Sensor Features into Full-run and Elution Basetables

Divides `full_sensor_features` into two separate feature sets:
- **Full-run**: `sensor_*` columns — statistics across the entire batch window per stage
- **Elution**: `elution_*` columns — statistics only during the elution phase per stage

Both sets keep the events duration features (`ev_*`) and elution availability flags.
SAP merging is done separately in the basetable notebook.


In [15]:
# Identify column groups by prefix
fullrun_cols = [c for c in full_sensor_features.columns
                if any(c.startswith(f'{s}_sensor_') for s in ['pg', 'deae', 'prep'])]

elution_cols = [c for c in full_sensor_features.columns
                if any(c.startswith(f'{s}_elution_') for s in ['pg', 'deae', 'prep'])]

flag_cols    = [c for c in full_sensor_features.columns
                if c in {'pg_has_elution_data', 'deae_has_elution_data'}]

ev_cols = [c for c in full_sensor_features.columns
           if (
               c.startswith('ev_') or
               any(c.startswith(f'{s}_{l}_')
                   for s in ['pg', 'deae'] for l in ['child1', 'child2']) or
               any(c.startswith(f'{s}_sub_')
                   for s in ['pg', 'deae'])
           )
           and c != 'ev_batch_id']

# Breakdown so the count is clear
child2_duration_cols    = [c for c in ev_cols if '_child2_' in c]
child2_sensor_ev_cols   = [c for c in ev_cols if '_sub_' in c and 'sensor_' in c]

print(f'Key column:           1')
print(f'Full-run sensor cols: {len(fullrun_cols)}')
print(f'Elution sensor cols:  {len(elution_cols)}')
print(f'Elution flag cols:    {len(flag_cols)}')
print(f'  Events duration cols (ev_*):  '
      f'{len([c for c in ev_cols if c.startswith("ev_")])}')
print(f'  Child 1 duration cols:        '
      f'{len([c for c in ev_cols if "_child1_" in c])}')
print(f'  Child 2 duration cols:        {len(child2_duration_cols)}')
print(f'  Child 2 sub-event sensor cols:{len(child2_sensor_ev_cols)}')
print(f'Total ev_cols:        {len(ev_cols)}')

# Build the two feature sets — both share the key, flags, and events features
sensor_fullrun = full_sensor_features[['ev_batch_id'] + fullrun_cols + flag_cols + ev_cols].copy()
sensor_elution = full_sensor_features[['ev_batch_id'] + elution_cols + flag_cols + ev_cols].copy()

print(f'\nFull-run feature set shape: {sensor_fullrun.shape}')
print(f'Elution feature set shape:  {sensor_elution.shape}')

# Check coverage
fullrun_coverage = sensor_fullrun[fullrun_cols].notna().any(axis=1).sum()
elution_coverage = sensor_elution[elution_cols].notna().any(axis=1).sum()
print(f'\nBatches with any full-run data: {fullrun_coverage} / {len(sensor_fullrun)}')
print(f'Batches with any elution data:  {elution_coverage} / {len(sensor_elution)}')

# Save both
fullrun_path = OUTPUT_DIR / 'sensor_features_fullrun_v5.parquet'
elution_path = OUTPUT_DIR / 'sensor_features_elution_v5.parquet'

sensor_fullrun.to_parquet(fullrun_path, index=False)
sensor_elution.to_parquet(elution_path, index=False)

print(f'\nSaved: {fullrun_path}')
print(f'Saved: {elution_path}')

Key column:           1
Full-run sensor cols: 264
Elution sensor cols:  264
Elution flag cols:    2
  Events duration cols (ev_*):  25
  Child 1 duration cols:        4
  Child 2 duration cols:        345
  Child 2 sub-event sensor cols:6248
Total ev_cols:        6622

Full-run feature set shape: (206, 6889)
Elution feature set shape:  (206, 6889)

Batches with any full-run data: 206 / 206
Batches with any elution data:  203 / 206

Saved: C:\Hackathon-GSK\data\processed\sensor_features_fullrun_v5.parquet
Saved: C:\Hackathon-GSK\data\processed\sensor_features_elution_v5.parquet


## Cell 13 — Build Final Basetables (Threshold + Type-Aware Imputation)

Merges SAP IP1/IP2/IP3 with `sensor_fullrun`. First applies an **80% missing-value
threshold** (`MISSING_THRESHOLD = 0.80`) per column group, then applies type-aware
imputation to the surviving columns:

- **Threshold step**: drops any column where more than 80% of values are NaN
  across the merged batch population (applied separately to SAP cols, full-run
  sensor cols, elution cols, child2 sub-event sensor cols, and duration cols)
- **Full-run sensor** (pg_sensor_*, deae_sensor_*, prep_sensor_*): column-mean imputation
  within each serotype + `sensor_data_imputed` flag for fully-missing batches
- **Child 2 sub-event sensor** (*_sub_*_sensor_*): zero-fill +
  one `{prefix}_ran` binary flag per sub-step type (1 = step ran for this batch)
- **Elution sensor** (*_elution_*): zero-fill (no peak = 0 absorbance)
- **Duration / event features** (ev_*, child1_, child2_): zero-fill
  (NaN = event did not occur = 0 hours duration)

In [16]:
MISSING_THRESHOLD = 0.80  # drop columns where NaN% across merged rows exceeds this

# Drop 2 known problematic batches from IP1
ids_to_drop = [
    '1_4add40cc92f02105db744984035bda8e980cce046b837d8ccbe6bab8491f9c10_',
    '1_0140125d488d704818ee66ff0785862ff63123a6089e73873e60edcee53bd914_',
]
ip1_clean = ip1[~ip1['Batch'].isin(ids_to_drop)].copy()
print(f'IP1: dropped {len(ids_to_drop)} problematic batches → {len(ip1_clean)} rows remain')

# ── Column group classification ──────────────────────────────────────────
# Computed from sensor_fullrun.columns so it always reflects the actual run.

fullrun_sensor_cols = [c for c in sensor_fullrun.columns
                       if c != 'ev_batch_id' and
                       any(c.startswith(f'{s}_sensor_') for s in ['pg', 'deae', 'prep'])]

child2_sensor_cols  = [c for c in sensor_fullrun.columns
                       if c != 'ev_batch_id' and '_sub_' in c and 'sensor_' in c]

child2_substep_prefixes = sorted(set(
    c.split('_sensor_')[0]
    for c in child2_sensor_cols
))

elution_sensor_cols = [c for c in sensor_fullrun.columns
                       if c != 'ev_batch_id' and '_elution_' in c]

ev_duration_cols    = [c for c in sensor_fullrun.columns
                       if c != 'ev_batch_id' and
                       c not in fullrun_sensor_cols + child2_sensor_cols +
                               elution_sensor_cols and
                       (c.startswith('ev_') or '_child1_' in c or '_child2_' in c or
                        c in {'pg_has_elution_data', 'deae_has_elution_data',
                              'pg_sensor_data_missing', 'deae_sensor_data_missing'})]

print('Column groups (before threshold):')
print(f'  Full-run sensor cols  : {len(fullrun_sensor_cols)}')
print(f'  Child2 sensor cols    : {len(child2_sensor_cols)}')
print(f'  Child2 sub-step types : {len(child2_substep_prefixes)}')
print(f'  Elution sensor cols   : {len(elution_sensor_cols)}')
print(f'  Duration / event cols : {len(ev_duration_cols)}')

# ── Per-serotype threshold stats (populated by build_basetable_v4 for Cell 15) ──
_threshold_stats = {}


def _drop_sparse(merged, cols, group_name, label, threshold):
    avail = [c for c in cols if c in merged.columns]
    if not avail:
        print(f'    {label} [{group_name}]: 0 cols present — skipped')
        return avail, 0
    nan_frac = merged[avail].isna().mean()
    to_drop  = nan_frac[nan_frac > threshold].index.tolist()
    if to_drop:
        merged.drop(columns=to_drop, inplace=True)
    kept = [c for c in avail if c not in to_drop]
    print(f'    {label} [{group_name}]: kept {len(kept)}, dropped {len(to_drop)} '
          f'(>{threshold*100:.0f}% NaN)')
    return kept, len(to_drop)


def build_basetable_v4(sap_df, sensor_df, label):
    merged = sap_df.merge(
        sensor_df,
        left_on='Batch',
        right_on='ev_batch_id',
        how='left'
    ).drop(columns=['ev_batch_id'], errors='ignore')

    cols_before = merged.shape[1]

    # ── STEP 1: Drop sparse columns (MISSING_THRESHOLD) ─────────────────────────
    print(f'\n  [{label}] Applying {MISSING_THRESHOLD*100:.0f}% missing-value threshold:')

    # SAP columns: from sap_df, excluding the join key and Reference Date
    sap_feature_cols = [c for c in sap_df.columns
                        if c not in ('Batch', 'Reference Date') and c in merged.columns]
    _, sap_dropped = _drop_sparse(merged, sap_feature_cols,
                                  'SAP', label, MISSING_THRESHOLD)

    avail_fullrun, fullrun_dropped = _drop_sparse(merged, fullrun_sensor_cols,
                                                  'fullrun_sensor', label, MISSING_THRESHOLD)

    avail_elution, elution_dropped = _drop_sparse(merged, elution_sensor_cols,
                                                  'elution_sensor', label, MISSING_THRESHOLD)

    avail_child2, child2_dropped  = _drop_sparse(merged, child2_sensor_cols,
                                                 'child2_sensor', label, MISSING_THRESHOLD)

    # Recompute prefixes after some child2 cols may have been dropped
    surviving_prefixes = sorted(set(
        c.split('_sensor_')[0] for c in avail_child2
    ))

    avail_ev, ev_dropped = _drop_sparse(merged, ev_duration_cols,
                                        'ev_duration', label, MISSING_THRESHOLD)

    cols_after_threshold = merged.shape[1]
    total_dropped = cols_before - cols_after_threshold
    _threshold_stats[label] = {
        'cols_before':      cols_before,
        'sap_dropped':      sap_dropped,
        'fullrun_dropped':  fullrun_dropped,
        'elution_dropped':  elution_dropped,
        'child2_dropped':   child2_dropped,
        'ev_dropped':       ev_dropped,
        'total_dropped':    total_dropped,
    }
    print(f'  [{label}] Total dropped: {total_dropped} / {cols_before} '
          f'({total_dropped/cols_before*100:.1f}%)')

    # ── STEP 2: Type-aware imputation on surviving columns ────────────────────
    print(f'  [{label}] Imputing:')

    # Type 1/2: full-run sensor — serotype-specific mean + sensor_data_imputed flag
    if avail_fullrun:
        merged['sensor_data_imputed'] = (
            merged[avail_fullrun].isna().all(axis=1).astype(int)
        )
        n_imputed = merged['sensor_data_imputed'].sum()
        col_means = merged[avail_fullrun].mean(numeric_only=True)
        merged[avail_fullrun] = merged[avail_fullrun].fillna(col_means)
        print(f'    {label}: {n_imputed}/{len(merged)} batches fully imputed '
              f'(sensor_data_imputed=1)')
    else:
        merged['sensor_data_imputed'] = 0

    # Type 3: child2 sub-event sensor — _ran flag per surviving prefix + zero-fill
    if avail_child2:
        for prefix in surviving_prefixes:
            prefix_cols = [c for c in avail_child2 if c.startswith(prefix + '_sensor_')]
            if prefix_cols:
                merged[prefix + '_ran'] = (
                    merged[prefix_cols].notna().any(axis=1).astype(int)
                )
        merged[avail_child2] = merged[avail_child2].fillna(0.0)
        ran_cols_added = len([c for c in merged.columns if c.endswith('_ran') and '_sub_' in c])
        print(f'    {label}: {ran_cols_added} _ran flags added, '
              f'{len(avail_child2)} child2 sensor cols zero-filled')

    # Type 4: elution sensor — zero-fill
    if avail_elution:
        merged[avail_elution] = merged[avail_elution].fillna(0.0)

    # Type 5: duration / event features — zero-fill
    if avail_ev:
        merged[avail_ev] = merged[avail_ev].fillna(0.0)

    # Safety net
    remaining_nan = merged.select_dtypes(include='number').isna().sum().sum()
    if remaining_nan > 0:
        print(f'    WARNING: {remaining_nan} NaNs remain — filling with 0 as fallback')
        merged = merged.fillna(0)

    return merged


print('\n=== Building basetables (threshold + type-aware imputation) ===')
ip1_bt = build_basetable_v4(ip1_clean, sensor_fullrun, 'IP1')
ip2_bt = build_basetable_v4(ip2,       sensor_fullrun, 'IP2')
ip3_bt = build_basetable_v4(ip3,       sensor_fullrun, 'IP3')

print()
for label, bt in [('IP1', ip1_bt), ('IP2', ip2_bt), ('IP3', ip3_bt)]:
    ran_cols = [c for c in bt.columns if c.endswith('_ran') and '_sub_' in c]
    print(f'  {label}: shape={bt.shape} | '
          f'sensor_data_imputed_count={bt["sensor_data_imputed"].sum()} | '
          f'_ran flag cols={len(ran_cols)} | '
          f'remaining_NaNs={bt.select_dtypes(include="number").isna().sum().sum()}')

IP1: dropped 2 problematic batches → 88 rows remain
Column groups (before threshold):
  Full-run sensor cols  : 264
  Child2 sensor cols    : 6248
  Child2 sub-step types : 71
  Elution sensor cols   : 2
  Duration / event cols : 374

=== Building basetables (threshold + type-aware imputation) ===

  [IP1] Applying 80% missing-value threshold:
    IP1 [SAP]: kept 145, dropped 0 (>80% NaN)
    IP1 [fullrun_sensor]: kept 264, dropped 0 (>80% NaN)
    IP1 [elution_sensor]: kept 2, dropped 0 (>80% NaN)
    IP1 [child2_sensor]: kept 6248, dropped 0 (>80% NaN)
    IP1 [ev_duration]: kept 369, dropped 5 (>80% NaN)
  [IP1] Total dropped: 5 / 7035 (0.1%)
  [IP1] Imputing:
    IP1: 0/88 batches fully imputed (sensor_data_imputed=1)
    IP1: 71 _ran flags added, 6248 child2 sensor cols zero-filled

  [IP2] Applying 80% missing-value threshold:
    IP2 [SAP]: kept 131, dropped 0 (>80% NaN)
    IP2 [fullrun_sensor]: kept 264, dropped 0 (>80% NaN)
    IP2 [elution_sensor]: kept 2, dropped 0 (>80% Na

## Cell 13b — PCA Reduction of Child 2 Sub-Event Sensor Features

The Child 2 sub-event sensor columns (*_sub_*_sensor_*) produce ~7,216 columns
for ~88 batches — an extreme high-dimensional regime where most models will overfit
without aggressive regularization. Per the professor's recommendation, PCA is applied
to compress this block into a small number of orthogonal components.

**Which columns are reduced:**
- `pg_sub_*_sensor_*` — PG-stage Child 2 sub-event sensor stats
- `deae_sub_*_sensor_*` — DEAE-stage Child 2 sub-event sensor stats

**Which columns are NOT reduced:**
- Full-run sensor cols (`pg_sensor_*`, `deae_sensor_*`, `prep_sensor_*`) — kept interpretable
- Elution sensor cols (`*_elution_*`) — physically meaningful signal
- Event duration cols (`ev_*`, `*_child1_*`, `*_child2_*`) — process timing, kept as-is
- SAP columns — never modified

**Two separate PCAs:** one PCA per stage (PG vs DEAE). The processes are physically
distinct; mixing them would conflate unrelated variance.

**Fit strategy:** PCA is fit on IP1 (largest dataset, 88 rows), then the same fitted
scaler + PCA is applied to IP2 and IP3. All three basetables share the same PC axes,
which is required for consistent model training across serotypes.

> **Data leakage note:** PCA is fit on all 206 batches before train/test split.
> Acceptable for proof-of-concept. The fitted `pca_models/*.joblib` files are saved
> so they can be applied to new batches in the dashboard without re-fitting.

**After this cell:** `ip1_bt`, `ip2_bt`, `ip3_bt` each lose ~7,000 sub-event sensor
columns and gain 10 new columns (`pg_sub_pc1`...`pg_sub_pc5`,
`deae_sub_pc1`...`deae_sub_pc5`).

In [17]:
N_PCA_COMPONENTS = 5

# ── Identify sub-event sensor columns per serotype ───────────────────────
def _get_sub_cols(bt, prefix):
    return [c for c in bt.columns if c.startswith(prefix) and '_sensor_' in c]

all_bts = [('IP1', ip1_bt), ('IP2', ip2_bt), ('IP3', ip3_bt)]

pg_sub_per   = {nm: _get_sub_cols(bt, 'pg_sub_')   for nm, bt in all_bts}
deae_sub_per = {nm: _get_sub_cols(bt, 'deae_sub_') for nm, bt in all_bts}

# Common columns across all three — consistent PCA feature space
pg_sub_cols   = sorted(set(pg_sub_per['IP1']) & set(pg_sub_per['IP2']) & set(pg_sub_per['IP3']))
deae_sub_cols = sorted(set(deae_sub_per['IP1']) & set(deae_sub_per['IP2']) & set(deae_sub_per['IP3']))

n_pg_sub_cols   = len(pg_sub_per['IP1'])
n_deae_sub_cols = len(deae_sub_per['IP1'])

print(f'PG   sub-event sensor cols in IP1 : {n_pg_sub_cols}')
print(f'     common across IP1/IP2/IP3   : {len(pg_sub_cols)}')
print(f'DEAE sub-event sensor cols in IP1 : {n_deae_sub_cols}')
print(f'     common across IP1/IP2/IP3   : {len(deae_sub_cols)}')

pca_dir = OUTPUT_DIR / 'pca_models'
pca_dir.mkdir(parents=True, exist_ok=True)

shapes_before = {nm: bt.shape for nm, bt in all_bts}
print('\nShapes before PCA:')
for nm, sh in shapes_before.items():
    print(f'  {nm}: {sh}')


def _apply_pca(all_bts, common_cols, prefix, n_comp):
    if not common_cols:
        print(f'  [{prefix}] No common columns across serotypes — PCA skipped.')
        return None, None, None

    n_comp_actual = min(n_comp, len(common_cols), min(len(bt) for _, bt in all_bts))

    # Fit scaler + PCA on IP1 only
    X0 = all_bts[0][1][common_cols].values
    scaler = StandardScaler()
    X0_sc  = scaler.fit_transform(X0)
    pca    = PCA(n_components=n_comp_actual, random_state=42)
    pca.fit(X0_sc)

    evr = pca.explained_variance_ratio_
    cum = np.cumsum(evr)
    pc_names = [f'{prefix}_pc{k+1}' for k in range(n_comp_actual)]

    print(f'  [{prefix}] {len(common_cols)} cols -> {n_comp_actual} PCs:')
    for k, (v, cv) in enumerate(zip(evr, cum)):
        print(f'    PC{k+1}: {v*100:.1f}%  (cumulative {cv*100:.1f}%)')

    for nm, bt in all_bts:
        # Drop ALL sub-event sensor cols for this prefix from this basetable
        all_prefix_cols = [c for c in bt.columns if c.startswith(prefix) and '_sensor_' in c]

        # Build aligned matrix using common_cols (zero-fill any missing col)
        X = np.zeros((len(bt), len(common_cols)))
        for j, c in enumerate(common_cols):
            if c in bt.columns:
                X[:, j] = bt[c].values
        X_sc = scaler.transform(X)
        X_pc = pca.transform(X_sc)

        bt.drop(columns=all_prefix_cols, inplace=True)
        for k, col in enumerate(pc_names):
            bt[col] = X_pc[:, k]

        print(f'    {nm}: {len(all_prefix_cols)} cols dropped, {n_comp_actual} PCs added, '
              f'shape={bt.shape}')

    return scaler, pca, evr


print('\n=== PCA: PG sub-event sensor columns ===')
pg_scaler, pg_pca, pg_pca_evr = _apply_pca(all_bts, pg_sub_cols, 'pg_sub', N_PCA_COMPONENTS)

print('\n=== PCA: DEAE sub-event sensor columns ===')
deae_scaler, deae_pca, deae_pca_evr = _apply_pca(all_bts, deae_sub_cols, 'deae_sub', N_PCA_COMPONENTS)

# Save fitted objects
if pg_scaler is not None:
    joblib.dump(pg_scaler, pca_dir / 'pg_sub_scaler.joblib')
    joblib.dump(pg_pca,    pca_dir / 'pg_sub_pca.joblib')
if deae_scaler is not None:
    joblib.dump(deae_scaler, pca_dir / 'deae_sub_scaler.joblib')
    joblib.dump(deae_pca,    pca_dir / 'deae_sub_pca.joblib')

print(f'\nPCA models saved to: {pca_dir}')
for f in sorted(pca_dir.glob('*.joblib')):
    print(f'  {f.name}')

print('\n=== PCA Reduction Summary ===')
for nm, bt in all_bts:
    sh_b = shapes_before[nm]
    print(f'  {nm}: {sh_b} -> {bt.shape}  '
          f'(removed {sh_b[1] - bt.shape[1]} cols)')


PG   sub-event sensor cols in IP1 : 3872
     common across IP1/IP2/IP3   : 3432
DEAE sub-event sensor cols in IP1 : 2376
     common across IP1/IP2/IP3   : 2376

Shapes before PCA:
  IP1: (88, 7102)
  IP2: (30, 6643)
  IP3: (87, 7099)

=== PCA: PG sub-event sensor columns ===
  [pg_sub] 3432 cols -> 5 PCs:
    PC1: 62.7%  (cumulative 62.7%)
    PC2: 6.3%  (cumulative 69.0%)
    PC3: 3.5%  (cumulative 72.5%)
    PC4: 2.8%  (cumulative 75.3%)
    PC5: 2.2%  (cumulative 77.5%)
    IP1: 3872 cols dropped, 5 PCs added, shape=(88, 3235)
    IP2: 3432 cols dropped, 5 PCs added, shape=(30, 3216)
    IP3: 3872 cols dropped, 5 PCs added, shape=(87, 3232)

=== PCA: DEAE sub-event sensor columns ===
  [deae_sub] 2376 cols -> 5 PCs:
    PC1: 28.3%  (cumulative 28.3%)
    PC2: 14.2%  (cumulative 42.5%)
    PC3: 9.0%  (cumulative 51.5%)
    PC4: 6.3%  (cumulative 57.8%)
    PC5: 5.3%  (cumulative 63.0%)
    IP1: 2376 cols dropped, 5 PCs added, shape=(88, 864)
    IP2: 2376 cols dropped, 5 PCs added,

## Cell 14 — Save Final Basetables

Saves 3 parquet files to `OUTPUT_DIR/basetable_v4/`.
Type-aware imputation is baked in — no separate “imputed” variant needed.

In [18]:
# Create output directory for basetables
bt_dir = OUTPUT_DIR / 'basetable_v5'
bt_dir.mkdir(parents=True, exist_ok=True)

files_to_save = {
    'ip1_basetable_v5.parquet': ip1_bt,
    'ip2_basetable_v5.parquet': ip2_bt,
    'ip3_basetable_v5.parquet': ip3_bt,
}

print(f'Saving to: {bt_dir}\n')
for fname, df in files_to_save.items():
    path = bt_dir / fname
    df.to_parquet(path, index=False)
    size_kb = path.stat().st_size / 1024
    print(f'  {fname:<40s}  shape={str(df.shape):<15s}  {size_kb:>8,.0f} KB')

print(f'\nAll {len(files_to_save)} basetable files saved.')

Saving to: C:\Hackathon-GSK\data\processed\basetable_v5

  ip1_basetable_v5.parquet                  shape=(88, 864)             856 KB
  ip2_basetable_v5.parquet                  shape=(30, 845)             682 KB
  ip3_basetable_v5.parquet                  shape=(87, 861)             859 KB

All 3 basetable files saved.


## Cell 15 — Pipeline Summary Report

Prints a full summary of the pipeline: input sizes, join results, feature dimensions,
basetable coverage per serotype, and paths of all saved files.

In [23]:
print('=' * 70)
print('PIPELINE SUMMARY — Time Series Merging v4 (DuckDB range join)')
print('=' * 70)

print(f'\n[Input Data]')
print(f'  XTO sensor files:             {len(sensor_files)}')
print(f'  Events files:                 {len(events_files)}')
print(f'  Valid SAP batch IDs:          {len(VALID_BATCH_IDS)}')

print(f'\n[DuckDB Range Join Results — Top-Level Batch Windows]')
print(f'  Sensor rows read (7 files):   {total_rows:>12,}')
print(f'  Rows after range join:        {total_assigned:>12,}')
match_rate = total_assigned / total_rows * 100
print(f'  Match rate:                   {match_rate:.1f}%')

print(f'\n[Event Time Windows (valid batches only)]')
ev_stage_counts = xto_map['stage'].value_counts()
print(f'  Total windows:                {len(xto_map):,}')
for stage in ['PG', 'DEAE', 'PREP']:
    cnt = ev_stage_counts.get(stage, 0)
    batches = xto_map[xto_map['stage'] == stage]['BatchID'].nunique()
    print(f'    {stage:<8s}: {cnt:>6,} windows across {batches} batches')

print(f'\n[Feature Set Dimensions]')
print(f'  Full-run sensor cols:         {len(fullrun_cols)}')
print(f'  Elution sensor cols:          {len(elution_cols)}')
print(f'  Events duration cols (total): {len(ev_cols)}')
print(f'  Elution flag cols:            {len(flag_cols)}')
print(f'  Full-run set shape:           {sensor_fullrun.shape}')
print(f'  Elution set shape:            {sensor_elution.shape}')

child1_cols = [c for c in full_sensor_features.columns if '_child1_' in c]
child2_cols = [c for c in full_sensor_features.columns if '_child2_' in c and '_sub_' not in c]
print(f'\n[Sub-Event Duration Features (Cell 5b)]')
print(f'  Child 1 duration cols: {len(child1_cols)}')
print(f'  Child 2 duration cols: {len(child2_cols)}')
print(f'  Sample Child 1 cols:   {child1_cols[:3]}')
print(f'  Sample Child 2 cols:   {child2_cols[:3]}')

child2_sensor_feat_cols = [c for c in full_sensor_features.columns
                           if '_sub_' in c and 'sensor_' in c]
n_substep_groups = child2_sensor_wide.shape[1] - 1 if not child2_sensor_wide.empty else 0
print(f'\n[Child 2 Sub-Event Sensor Features (new in v4)]')
print(f'  Sub-step sensor columns:      {len(child2_sensor_feat_cols)}')
if len(child2_sensor_feat_cols) > 0:
    cov = full_sensor_features[child2_sensor_feat_cols].notna().any(axis=1).sum()
    print(f'  Batch coverage:               {cov} / {len(full_sensor_features)}')
    print(f'  Distinct sub-step groups:     '
          f'{len(set(c.rsplit("_sensor_", 1)[0] for c in child2_sensor_feat_cols))}')
    print(f'  Sample cols: {child2_sensor_feat_cols[:3]}')
else:
    print('  (none produced — child2_xto_map was empty)')

print(f'\n[Missing Value Threshold ({MISSING_THRESHOLD*100:.0f}%) — Cell 13 Column Drops]')
for label, bt in [('IP1', ip1_bt), ('IP2', ip2_bt), ('IP3', ip3_bt)]:
    if label in _threshold_stats:
        s = _threshold_stats[label]
        pct = s['total_dropped'] / s['cols_before'] * 100
        print(f'  {label}: {s["cols_before"]} cols before '
              f'→ {s["total_dropped"]} dropped ({pct:.1f}%) '
              f'→ {bt.shape[1]} final cols')
        print(f'       SAP={s["sap_dropped"]}  fullrun={s["fullrun_dropped"]}  '
              f'elution={s["elution_dropped"]}  child2={s["child2_dropped"]}  '
              f'ev_dur={s["ev_dropped"]}')

print(f'\n[Final Basetable Coverage]')
sensor_feature_cols = [c for c in sensor_fullrun.columns if c != 'ev_batch_id']
for label, bt in [('IP1', ip1_bt), ('IP2', ip2_bt), ('IP3', ip3_bt)]:
    # Intersect with bt.columns — sub-event sensor cols were replaced by PCs in Cell 13b
    avail_sensor = [c for c in sensor_feature_cols if c in bt.columns]
    sap_only = bt[avail_sensor].isna().all(axis=1).sum() if avail_sensor else 0
    with_sensor = len(bt) - sap_only
    pct = with_sensor / len(bt) * 100
    print(f'  {label}: {len(bt):3d} batches total | '
          f'{with_sensor} with sensor data ({pct:.0f}%) | {sap_only} SAP-only')

print(f'\n[Saved Files — basetable_v4/]')
for fname in files_to_save:
    p = bt_dir / fname
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f'  {fname:<45s}  {size_mb:.2f} MB')

print(f'\n[Intermediate Sensor Files]')
for fname in ['sensor_features_fullrun_v4.parquet',
              'sensor_features_elution_v4.parquet',
              'sensor_aggregated_features_v4.parquet']:
    p = OUTPUT_DIR / fname
    if p.exists():
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f'  {fname:<45s}  {size_mb:.2f} MB')

print(f'\n[PCA Reduction -- Child 2 Sub-Event Sensor Features]')
if pg_pca_evr is not None:
    cum_pg = float(np.sum(pg_pca_evr)) * 100
    print(f'  PG   sub-event: {n_pg_sub_cols} cols -> 5 PCs '
          f'(cumulative variance explained: {cum_pg:.1f}%)')
if deae_pca_evr is not None:
    cum_deae = float(np.sum(deae_pca_evr)) * 100
    print(f'  DEAE sub-event: {n_deae_sub_cols} cols -> 5 PCs '
          f'(cumulative variance explained: {cum_deae:.1f}%)')
print(f'  PCA models saved to: {pca_dir}')

print('\n' + '=' * 70)
print('Pipeline complete')
print('=' * 70)



PIPELINE SUMMARY — Time Series Merging v4 (DuckDB range join)

[Input Data]
  XTO sensor files:             7
  Events files:                 7
  Valid SAP batch IDs:          207

[DuckDB Range Join Results — Top-Level Batch Windows]
  Sensor rows read (7 files):     44,102,249
  Rows after range join:           7,536,494
  Match rate:                   17.1%

[Event Time Windows (valid batches only)]
  Total windows:                828
    PG      :    204 windows across 204 batches
    DEAE    :    205 windows across 205 batches
    PREP    :    419 windows across 207 batches

[Feature Set Dimensions]
  Full-run sensor cols:         264
  Elution sensor cols:          264
  Events duration cols (total): 6622
  Elution flag cols:            2
  Full-run set shape:           (206, 6889)
  Elution set shape:            (206, 6889)

[Sub-Event Duration Features (Cell 5b)]
  Child 1 duration cols: 4
  Child 2 duration cols: 345
  Sample Child 1 cols:   ['pg_child1_op13_user_request_2_1',